# 🎮 DQN Breakout — Pretty (Lower Hyperparameters)
**Environment:** `ALE/Breakout-v5` | **Timesteps:** 500,000 | **Policy:** CnnPolicy

| Param | Range |
|---|---|
| lr | 0.00001 – 0.00005 |
| gamma | 0.90 – 0.95 |
| batch | 16 – 32 |
| eps_end | 0.01 – 0.03 |
| eps_frac | 0.05 – 0.10 |

## Step 1 — Install Dependencies

In [ ]:
!pip install stable-baselines3[extra] gymnasium[atari] ale-py autorom -q
!AutoROM --accept-license

## Step 2 — Write train.py to disk

In [ ]:
%%writefile train.py
"""
================================================
  train.py — DQN Breakout
  Group Assignment — Unified Training Script
  Members: Erneste, Victoria, Pretty
  Stable Baselines 3 + Gymnasium
================================================
"""

import os
import csv
import json
import time
import argparse
import numpy as np
import ale_py
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecTransposeImage
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
    CallbackList,
)

gym.register_envs(ale_py)

ENV_ID = "ALE/Breakout-v5"
SEED   = 42


def parse_args():
    parser = argparse.ArgumentParser(description="Train one DQN Breakout experiment")
    parser.add_argument("--member",          type=str,   required=True)
    parser.add_argument("--experiment",      type=str,   required=True)
    parser.add_argument("--lr",              type=float, required=True)
    parser.add_argument("--gamma",           type=float, required=True)
    parser.add_argument("--batch",           type=int,   required=True)
    parser.add_argument("--eps-start",       type=float, default=1.0)
    parser.add_argument("--eps-end",         type=float, required=True)
    parser.add_argument("--eps-frac",        type=float, required=True)
    parser.add_argument("--total-timesteps", type=int,   default=500_000)
    parser.add_argument("--buffer-size",     type=int,   default=100_000)
    parser.add_argument("--optimize-memory", action="store_true")
    return parser.parse_args()


def make_env(seed):
    env = make_atari_env(ENV_ID, n_envs=1, seed=seed)
    env = VecFrameStack(env, n_stack=4)
    env = VecTransposeImage(env)
    return env


def main():
    args = parse_args()
    member  = args.member
    exp_tag = args.experiment
    tag     = exp_tag

    base_dir     = os.path.join("results", member)
    model_dir    = os.path.join(base_dir, "models")
    log_dir      = os.path.join(base_dir, "logs")
    exp_eval_dir = os.path.join(base_dir, exp_tag, "eval_logs")
    exp_ckpt_dir = os.path.join(base_dir, exp_tag, "checkpoints")

    for d in [model_dir, log_dir, exp_eval_dir, exp_ckpt_dir]:
        os.makedirs(d, exist_ok=True)

    config = {
        "member": member, "experiment": exp_tag, "env_id": ENV_ID,
        "lr": args.lr, "gamma": args.gamma, "batch": args.batch,
        "eps_start": args.eps_start, "eps_end": args.eps_end,
        "eps_frac": args.eps_frac,
        "total_timesteps": args.total_timesteps,
        "buffer_size": args.buffer_size,
    }
    with open(os.path.join(base_dir, exp_tag, f"{exp_tag}_config.json"), "w") as f:
        json.dump(config, f, indent=2)

    print("\n" + "=" * 65)
    print(f"  [{member}]  {exp_tag}")
    print(f"  lr={args.lr}  gamma={args.gamma}  batch={args.batch}")
    print(f"  eps_end={args.eps_end}  eps_frac={args.eps_frac}")
    print("=" * 65)

    train_env = make_env(SEED)
    eval_env  = make_env(SEED + 1)

    model = DQN(
        policy="CnnPolicy", env=train_env,
        learning_rate=args.lr, gamma=args.gamma, batch_size=args.batch,
        buffer_size=args.buffer_size, learning_starts=20_000,
        train_freq=4, target_update_interval=1_000,
        exploration_initial_eps=args.eps_start,
        exploration_final_eps=args.eps_end,
        exploration_fraction=args.eps_frac,
        optimize_memory_usage=args.optimize_memory,
        verbose=1, seed=SEED,
    )

    checkpoint_cb = CheckpointCallback(
        save_freq=250_000, save_path=exp_ckpt_dir,
        name_prefix=f"dqn_{tag}", verbose=0,
    )
    eval_cb = EvalCallback(
        eval_env, best_model_save_path=exp_eval_dir,
        log_path=exp_eval_dir, eval_freq=50_000,
        n_eval_episodes=20, deterministic=True, verbose=0,
    )

    start_time = time.time()
    model.learn(
        total_timesteps=args.total_timesteps,
        callback=CallbackList([checkpoint_cb, eval_cb]),
        reset_num_timesteps=True, progress_bar=False,
    )
    train_minutes = (time.time() - start_time) / 60.0

    final_model_path = os.path.join(model_dir, f"dqn_{tag}")
    model.save(final_model_path)

    eval_npz    = os.path.join(exp_eval_dir, "evaluations.npz")
    mean_reward = 0.0
    std_reward  = 0.0
    noted       = "No eval data."
    exp_csv     = os.path.join(log_dir, f"{tag}_reward_log.csv")

    if os.path.exists(eval_npz):
        data        = np.load(eval_npz)
        timesteps   = data["timesteps"]
        all_results = data["results"]
        all_lengths = data["ep_lengths"]
        final       = all_results[-1]
        mean_reward = float(np.mean(final))
        std_reward  = float(np.std(final))
        trend       = all_results.mean(axis=1)
        noted = (
            "Reward improving across training."           if trend[-1] > trend[0]
            else "Reward declined — possible instability." if trend[-1] < trend[0]
            else "Reward flat — may need more timesteps."
        )
        with open(exp_csv, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["timestep", "mean_reward", "std_reward", "mean_ep_length"])
            for t, r, l in zip(timesteps, all_results, all_lengths):
                writer.writerow([int(t), round(float(np.mean(r)), 2),
                                 round(float(np.std(r)), 2),
                                 round(float(np.mean(l)), 1)])

    eval_summary = {
        "member": member, "experiment": exp_tag, "env_id": ENV_ID,
        "mean_reward": round(mean_reward, 2), "std_reward": round(std_reward, 2),
        "train_minutes": round(train_minutes, 2), "noted": noted,
        "model_path": f"{final_model_path}.zip",
        "best_model_path": os.path.join(exp_eval_dir, "best_model.zip"),
        "training_csv_path": exp_csv,
        **config,
    }
    eval_json_path = os.path.join(base_dir, exp_tag, f"{exp_tag}_eval.json")
    with open(eval_json_path, "w") as f:
        json.dump(eval_summary, f, indent=2)

    train_env.close()
    eval_env.close()

    print(f"\n  ✓ {tag}  Mean Reward: {mean_reward:.2f}  Std: {std_reward:.2f}")
    print(f"  Train minutes: {train_minutes:.2f}")


if __name__ == "__main__":
    main()

## Step 3 — Setup: imports, paths, experiment definitions

### Pretty — Lower Hyperparameters
| Exp | lr | gamma | batch | eps_end | eps_frac |
|---|---|---|---|---|---|
| 1  | 0.00001 | 0.90 | 16 | 0.01 | 0.05 |
| 2  | 0.00001 | 0.92 | 16 | 0.01 | 0.05 |
| 3  | 0.00001 | 0.95 | 32 | 0.01 | 0.08 |
| 4  | 0.00003 | 0.90 | 16 | 0.01 | 0.05 |
| 5  | 0.00003 | 0.92 | 32 | 0.02 | 0.08 |
| 6  | 0.00003 | 0.95 | 32 | 0.02 | 0.08 |
| 7  | 0.00005 | 0.90 | 16 | 0.01 | 0.05 |
| 8  | 0.00005 | 0.92 | 32 | 0.02 | 0.08 |
| 9  | 0.00005 | 0.95 | 32 | 0.03 | 0.10 |
| 10 | 0.00003 | 0.95 | 16 | 0.02 | 0.10 |

In [ ]:
import sys
import subprocess
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

MEMBER          = "Pretty"
TOTAL_TIMESTEPS = 500_000
BASE_DIR        = Path("results") / MEMBER

experiments = [
    {"name": "exp01", "lr": 0.00001, "gamma": 0.90, "batch": 16, "eps_end": 0.01, "eps_frac": 0.05},
    {"name": "exp02", "lr": 0.00001, "gamma": 0.92, "batch": 16, "eps_end": 0.01, "eps_frac": 0.05},
    {"name": "exp03", "lr": 0.00001, "gamma": 0.95, "batch": 32, "eps_end": 0.01, "eps_frac": 0.08},
    {"name": "exp04", "lr": 0.00003, "gamma": 0.90, "batch": 16, "eps_end": 0.01, "eps_frac": 0.05},
    {"name": "exp05", "lr": 0.00003, "gamma": 0.92, "batch": 32, "eps_end": 0.02, "eps_frac": 0.08},
    {"name": "exp06", "lr": 0.00003, "gamma": 0.95, "batch": 32, "eps_end": 0.02, "eps_frac": 0.08},
    {"name": "exp07", "lr": 0.00005, "gamma": 0.90, "batch": 16, "eps_end": 0.01, "eps_frac": 0.05},
    {"name": "exp08", "lr": 0.00005, "gamma": 0.92, "batch": 32, "eps_end": 0.02, "eps_frac": 0.08},
    {"name": "exp09", "lr": 0.00005, "gamma": 0.95, "batch": 32, "eps_end": 0.03, "eps_frac": 0.10},
    {"name": "exp10", "lr": 0.00003, "gamma": 0.95, "batch": 16, "eps_end": 0.02, "eps_frac": 0.10},
]

BEST_CONFIG = {"name": "expBEST", "lr": 0.00005, "gamma": 0.95, "batch": 32,
               "eps_end": 0.02, "eps_frac": 0.08, "buffer_size": 50_000}

print(f"✓ {len(experiments)} experiments defined for {MEMBER}")
print(f"✓ Total timesteps per experiment: {TOTAL_TIMESTEPS:,}")

## Step 4 — run_experiment helper (calls train.py via subprocess)

In [ ]:
def run_experiment(exp: dict):
    cmd = [
        sys.executable, "train.py",
        "--member",          MEMBER,
        "--experiment",      exp["name"],
        "--lr",              str(exp["lr"]),
        "--gamma",           str(exp["gamma"]),
        "--batch",           str(exp["batch"]),
        "--eps-end",         str(exp["eps_end"]),
        "--eps-frac",        str(exp["eps_frac"]),
        "--total-timesteps", str(TOTAL_TIMESTEPS),
        "--buffer-size",     str(exp.get("buffer_size", 100_000)),
    ]
    if exp.get("optimize_memory", False):
        cmd.append("--optimize-memory")

    print("\nRunning:", " ".join(cmd))
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"Experiment {exp['name']} failed (exit {process.returncode})")

print("✓ run_experiment helper ready")

## Step 5 — Run all 10 experiments
> ⏱️ **Expected time:** ~4–8 hours. Use a GPU runtime for best results.

In [ ]:
for exp in experiments:
    run_experiment(exp)

print("\n✓ All 10 experiments complete!")

## Step 6 — Summary table

In [ ]:
rows = []
for exp in experiments:
    eval_path = BASE_DIR / exp["name"] / f"{exp['name']}_eval.json"
    if eval_path.exists():
        with open(eval_path) as f:
            rows.append(json.load(f))

df = pd.DataFrame(rows).sort_values("mean_reward", ascending=False).reset_index(drop=True)
print(df[["experiment", "lr", "gamma", "batch", "eps_end", "eps_frac",
          "mean_reward", "std_reward", "noted"]].to_string(index=False))

best = df.iloc[0]
print(f"\n  Best: {best['experiment']}  Mean Reward: {best['mean_reward']:.2f}")

## Step 7 — Run expBEST (optimised config)

In [ ]:
run_experiment({**BEST_CONFIG, "optimize_memory": True})
print("✓ expBEST done")

## Step 8 — Save best model as dqn_model.zip

In [ ]:
import shutil

all_exps = experiments + [BEST_CONFIG]
all_rows = []
for exp in all_exps:
    p = BASE_DIR / exp["name"] / f"{exp['name']}_eval.json"
    if p.exists():
        with open(p) as f:
            all_rows.append(json.load(f))

all_df      = pd.DataFrame(all_rows).sort_values("mean_reward", ascending=False)
overall     = all_df.iloc[0]
best_tag    = overall["experiment"]
best_src    = BASE_DIR / "models" / f"dqn_{best_tag}.zip"
best_dst    = BASE_DIR / "models" / "dqn_model.zip"

if best_src.exists():
    shutil.copy(best_src, best_dst)
    print(f"✓ Best model: {best_tag}  ({overall['mean_reward']:.2f})  → {best_dst}")
else:
    print(f"⚠ {best_src} not found")

## Step 9 — Plot reward curves

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()

for i, exp in enumerate(experiments):
    log_csv = BASE_DIR / "logs" / f"{exp['name']}_reward_log.csv"
    if log_csv.exists():
        data = pd.read_csv(log_csv)
        axes[i].plot(data["timestep"], data["mean_reward"], color="coral")
        axes[i].fill_between(data["timestep"],
                             data["mean_reward"] - data["std_reward"],
                             data["mean_reward"] + data["std_reward"],
                             alpha=0.2, color="coral")
        axes[i].set_title(f"{exp['name']} | lr={exp['lr']}\ngamma={exp['gamma']} batch={exp['batch']}",
                          fontsize=8)
    else:
        axes[i].set_title(f"{exp['name']} — no data", fontsize=8)
    axes[i].set_xlabel("Timestep", fontsize=7)
    axes[i].set_ylabel("Reward", fontsize=7)

plt.suptitle(f"{MEMBER} — Reward Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(BASE_DIR / "logs" / "reward_curves.png", dpi=150)
plt.show()
print("✓ Plot saved")